# Text generation - chain

In this Notebook, we will build a system to generate words based on their co-occurrence in a text.

## Legend

You will find the following icons throughout the text:

📝 You need to do something.

💭 A question to trigger your attention on an important point of detail.

💬 Something you might want to discuss.

💡 A tip or an idea for later

## Credits

Notebook jointly authored by [David Grellscheid](uib.no/en/persons/David.Günter.Grellscheid) and [Marc Vaudel](www.uib.no/en/persons/Marc.Vaudel). Licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0) to the authors.

## Set up

📝 Run the cell below to import the libraries needed throughout the notebook.

In [1]:
# Libraries needed throughout the Notebook
import random
import requests
from collections import Counter

## Reference data

For our system to provide meaningful suggestions, we use reference data, sometimes also called _training data_. We will use free books from the [Project Gutenberg](https://www.gutenberg.org/).

📝 In the code below, uncomment one or several books to use for training.

In [9]:
books = [
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/odyssey.txt", # The Odyssey by Homer translated by Samuel Butler
    "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/alice.txt", # Alice’s Adventures in Wonderland by Lewis Carroll
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/timemachine.txt", # The Time Machine by H. G. Wells
    "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/gatsby.txt", # The Great Gatsby by F. Scott Fitzgerald
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/beowulf.txt", # Beowulf: An Anglo-Saxon Epic Poem by J. Lesslie Hall
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/prince.txt", # The Prince by Niccolò Machiavelli translated by W. K. Marriott
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/persuasion.txt", # Persuasion by Jane Austen
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/mansfield.txt", # Mansfield Park by Jane Austen
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/emma.txt", # Emma by Jane Austen
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/pride.txt", # Pride and Prejudice by Jane Austen
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/northanger.txt", # Northanger Abbey by Jane Austen
    # "https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/sense.txt", # Sense and Sensibility by Jane Austen

]


💬 Are we allowed to use these books?

💬 How will the choice of books influence our predictions?


## Training predictor

In the code below, we parse the main text of each book, and store which word comes after which.


Example 1: _I love programming in python._ becomes
```
I -> love
love -> programming
programming -> in
in -> python
```

💬 How would such a text generator work for people programming in _R_?

Note how this looks like a chain? This can be described as a [Markov chain](https://en.wikipedia.org/wiki/Markov_chain). We will build such chains keeping track of one, two, and three words.

Example 2: _I love programming in python._ becomes
```
I, love -> programming
love, programming -> in
programming, in -> python
```

Example 3: _I love programming in python._ becomes
```
I, love, programming -> in
love, programming, in -> python
```

💬 What happens as we increase the number of words used prior to the next one?

💡 Tip: you would like to write something like this but don't know how to write it in your favorite programming language? Most likely someone else had a smiliar issue and you will find the solution on the internet. Chatbots like [chat.uib.no](chat.uib.no) can also provide very good suggestions to get started.


In [10]:

def get_text_from_url(url):
    """
    Gets text from a URL.
    """

    # Query the URL and save the response
    response = requests.get(url)

    # If the response is an error, raise an exception
    response.raise_for_status()

    # Get the content
    return response.text


def fetch_main_from_gutenberg(full_text):
    """
    Extract the main body of a Project Gutenberg text file, stripping the content before and after.
    """
    text = []
    in_main_text = False
    for line in full_text.splitlines():
        if line.startswith("*** START"):
            in_main_text = True
            continue

        if line.startswith("*** END"):
            in_main_text = False
            continue

        if in_main_text:
            text.append(line)

    return " ".join(text)


def fill_chain_1(chain, text):
    """
    Fill the markov chain dictionary "chain" from a source text.
    This function looks only at the single last word.
    """
    words = text.split()
    for i in range(len(words)-1):
        word = words[i]
        next = words[i+1]

        if word in chain:
            chain[word].append(next)
        else:
            chain[word] = [next]


def fill_chain_2(chain, text):
    """
    Fill the markov chain dictionary "chain" from a source text.
    This function looks at two preceding words.
    """
    words = text.split()
    for i in range(1, len(words)-1):
        prev = words[i-1]
        word = words[i]
        next = words[i+1]

        key = (prev,word)
        if key in chain:
            chain[key].append(next)
        else:
            chain[key] = [next]


def fill_chain_3(chain, text):
    """
    Fill the markov chain dictionary "chain" from a source text.
    This function looks at three preceding words.
    """
    words = text.split()
    for i in range(2, len(words)-1):
        pprev = words[i-2]
        prev = words[i-1]
        word = words[i]
        next = words[i+1]

        key = (pprev,prev,word)
        if key in chain:
            chain[key].append(next)
        else:
            chain[key] = [next]

chain_1 = {}
chain_2 = {}
chain_3 = {}

for book_url in books:
    print(f"Processing {book_url}")
    raw_text = get_text_from_url(book_url)
    text = fetch_main_from_gutenberg(raw_text)
    fill_chain_1(chain_1, text)
    fill_chain_2(chain_2, text)
    fill_chain_3(chain_3, text)

print("Model training is done")


Processing https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/alice.txt
Processing https://raw.githubusercontent.com/dgrell/datacafe-2024-04/main/proj_gutenberg_books/gatsby.txt
Model training is done


Now let's look at the most common words in our chains.

In [11]:
def print_chain_top(chain):
    """
    Print the eight longest entries from the markov chain dictionary in a human readable form.
    """
    top = sorted(chain.items(), key=lambda x: -len(x[1]))[:8]
    top = [f"{k} : {v[:5]+['...']}" for k,v in top]
    print()
    print(*top, sep="\n")
    print()

print("# Top of chain 1:")
print_chain_top(chain_1)

print("# Top of chain 2:")
print_chain_top(chain_2)

print("# Top of chain 3:")
print_chain_top(chain_3)

# Top of chain 1:

the : ['Rabbit-Hole', 'Tarts?', 'Rabbit-Hole', 'bank,', 'book', '...']
and : ['David', 'a', 'Pepper', 'of', 'stupid),', '...']
a : ['Long', 'Little', 'Caterpillar', 'book,”', 'daisy-chain', '...']
to : ['get', 'do:', 'hear', 'itself,', 'her', '...']
of : ['Tears', 'sitting', 'having', 'a', 'making', '...']
I : ['shall', 'shall', 'wouldn’t', 'fell', 'think—”', '...']
in : ['Wonderland', 'a', 'it,', 'her', 'that;', '...']
was : ['beginning', 'reading,', 'considering', 'nothing', 'just', '...']

# Top of chain 2:

('of', 'the') : ['way', 'well,', 'shelves', 'cupboards', 'house!”', '...']
('in', 'the') : ['world', 'schoolroom,', 'air,', 'lock,', 'common', '...']
('said', 'the') : ['Mouse', 'Lory,', 'Mouse,', 'Lory', 'Mouse.', '...']
('in', 'a') : ['Little', 'dreamy', 'moment:', 'long,', 'hurry.', '...']
('and', 'the') : ['fall', 'White', 'fan,', 'words', 'little', '...']
('on', 'the') : ['bank,', 'second', 'top', 'glass', 'English', '...']
('to', 'the') : ['table,', 'doo

💬 How is our model handling sentence structure and punctuation marks?


## Prediction of next word

In the code below, we compute the frequency of words following the combination _said the_.


In [12]:

prev, word = "said", "the"

# Get the words following this combination, return "Not found" if none.
words = chain_2.get((prev, word), ["Not found"])

# Get the frequency of each word in the results
frequency = Counter(words)

# print the 10 most frequent words
top_10_words = frequency.most_common(10)
for item, count in top_10_words:
    print(f'{item}: {count}')


Mock: 19
Caterpillar.: 12
King,: 10
King.: 10
Hatter.: 9
March: 8
Queen,: 7
Gryphon.: 7
Duchess,: 6
Cat,: 6


💬 If we want the word following _said the_, what is the most likely outcome?

Now let's sample a word from this list.

In [13]:
# Sample a word randomly from the output
next_word = random.choice(words)

print(f"input: {prev} {word}")
print(f"prediction: {prev} {word} {next_word}")

input: said the
prediction: said the White


📝 Feel free to repreat with other combinations of words and a different chain to see what happens.

💬 What would happen if we use another book?

## Sentence generation

In the following code, we predict one word after the other to generate a sentence using chain_3.

💭 based on your training data, do you expect a predition on programming in python?

In [14]:
pprev, prev, word = "I", "love", "programming"

print(f"Starting a chain on '{pprev} {prev} {word}'")
print("====")
print(f"{pprev} {prev} {word}", end=" ")

generated_words = 1
while not word.endswith("."):

    if generated_words % 12 == 0:
        # line break every 12 words for readability
        print()

    # see if the word is in our training data, if not, try to predict from the previous word or use a random word.
    if not word in chain_1:
        if prev in chain_1:
            word = chain_1[prev][0]
        else:
            word = random.choice(list(chain_1.keys()))

    # sample a word from chain_1
    cand_1 = random.choice(chain_1[word])

    # see if we have a result with chain_2, if not, fall back to the result of chain_1
    cand_2 = random.choice(chain_2.get((prev, word), [cand_1]))

    # see if we have a result with chain_3, if not, fall back to the result of chain_2
    cand_3 = random.choice(chain_3.get((pprev, prev, word), [cand_2]))

    # use the candidate proposed by your favorite chain
    next_word = cand_2

    # shift the words to query by one position
    pprev, prev, word = prev, word, next_word
    print(word, end=" ")

    generated_words += 1


print("\n====")

Starting a chain on 'I love programming'
====
I love programming me. 
====


📝 Change the chain used, how does this affect the results?

## Text generation

Below, we extend the previous code to sample from the three chains with different probabilities until reaching a given length.

💭 Do you see the difference with the previous code?

In [15]:
pprev, prev, word = "I", "love", "programming"

weight_chain_1 = 4
weight_chain_2 = 16
weight_chain_3 = 80

print(f"Starting a chain on '{pprev} {prev} {word}'")
print("====")
print(f"{pprev} {prev} {word}", end=" ")

remaining_output = 200
while remaining_output:

    if remaining_output % 12 == 0:
        # line break every 12 words for readability
        print()

    # see if the word is in our training data, if not, try to predict from the previous word or use a random word.
    if not word in chain_1:
        if prev in chain_1:
            word = chain_1[prev][0]
        else:
            word = random.choice(list(chain_1.keys()))

    # sample a word from chain_1
    cand_1 = random.choice(chain_1[word])

    # see if we have a result with chain_2, if not, fall back to the result of chain_1
    cand_2 = random.choice(chain_2.get((prev, word), [cand_1]))

    # see if we have a result with chain_3, if not, fall back to the result of chain_2
    cand_3 = random.choice(chain_3.get((pprev, prev, word), [cand_2]))

    # use the candidate proposed by your favorite chain
    candidates = random.choices([cand_1, cand_2, cand_3], weights=[weight_chain_1, weight_chain_2, weight_chain_3])
    next_word = candidates[0]

    # shift the words to query by one position
    pprev, prev, word = prev, word, next_word
    print(word, end=" ")

    remaining_output -= 1


print("\n====")

Starting a chain on 'I love programming'
====
I love programming me. “Do they miss me?” she cried ecstatically. 
“The whole town is … Well, he’s no use in waiting by 
the little door, so she went on, taking first one side and 
then the owl-eyed man said “Amen to that,” in a moment; I 
wouldn’t have been surprised to see sinister faces, the faces of “Wolfshiem’s 
people,” behind him in the night. In a little while, however, she 
again heard a little animal (she couldn’t guess of what sort it 
was) scratching and scrambling about in the chimney close above her: then, 
saying to herself “That’s quite enough—I hope I shan’t grow any more—As 
it is, I can’t help what’s past.” She began to cry—she cried 
and cried. I rushed out and found her mother’s maid, and we 
locked the door and came out, his mouth ajar, his face pressed 
against it—“and I said ‘God knows what you’ve been thinking over what 
I have mattered then?” Silence for a moment. Then: “However—I want to 
see you, sir … Nick …” “M